# AI Equity Trading Agent

I extended Ed's trading simulator by

1. Giving each Trader direct `mcp-server-fetch` capability.  
Currently, only the Researcher sub-agent can fetch web pages. By adding `mcp-server-fetch` directly to the Trader's MCP server list, traders can independently retrieve a company's investor-relations page, an earnings press release URL, or any financial data endpoint without spinning up the full Researcher sub-agent for simple lookups.

2. Adding a Risk Manager as a second sub-agent tool on each Trader.  
Before executing a trade, the Trader can consult the Risk Manager, which checks three constraints:
1. No single position exceeds 25% of total portfolio value (concentration risk)
2. At least 10% of portfolio value stays in cash at all times (liquidity reserve)
3. No single trade deploys more than 15% of total portfolio value (trade sizing)

The Risk Manager returns `APPROVED`, `REDUCE` (with a suggested quantity), or `REJECTED` with a reason.  



In [ ]:
import json
import asyncio
import sys
import os
from contextlib import AsyncExitStack
from dotenv import load_dotenv

# Resolve the 6_mcp directory relative to this notebook
notebook_dir = os.getcwd()
six_mcp_dir = os.path.abspath(os.path.join(notebook_dir, "../../"))

# Change CWD to 6_mcp/ so all relative imports and MCP server paths work as-is
os.chdir(six_mcp_dir)

# Add 6_mcp/ to sys.path so Ed's local modules (reset, accounts, market, etc.) are importable
if six_mcp_dir not in sys.path:
    sys.path.insert(0, six_mcp_dir)

# Load .env from the project root (one level above 6_mcp/)
load_dotenv(dotenv_path=os.path.join(six_mcp_dir, "../.env"), override=True)

# Ensure uv/uvx (at ~/.local/bin) are on PATH for MCP subprocesses
os.environ["PATH"] = os.environ.get("PATH", "") + f":{os.path.expanduser('~/.local/bin')}"

# Massive.com is the 2025 rebrand of Polygon.io — APIs are 100% compatible.
# Bridge MASSIVE_API_KEY to POLYGON_API_KEY so Ed's market.py picks it up.
if not os.getenv("POLYGON_API_KEY") and os.getenv("MASSIVE_API_KEY"):
    os.environ["POLYGON_API_KEY"] = os.getenv("MASSIVE_API_KEY")
    print("Bridged MASSIVE_API_KEY to POLYGON_API_KEY")

# Default to free-tier (end-of-day prices) unless overridden in .env
os.environ.setdefault("POLYGON_PLAN", "free")

print(f"Notebook dir      : {notebook_dir}")
print(f"Working dir       : {os.getcwd()}")
print(f"POLYGON_API_KEY   : {'set' if os.getenv('POLYGON_API_KEY') else 'NOT SET'}")
print(f"BRAVE_API_KEY     : {'set' if os.getenv('BRAVE_API_KEY') else 'NOT SET'}")
print(f"POLYGON_PLAN      : {os.getenv('POLYGON_PLAN')}")

# OpenAI Agents SDK
from agents import Agent, Tool, Runner, trace, add_trace_processor
from agents.mcp import MCPServerStdio

# Ed's modules (all importable now that sys.path and env vars are set)
from reset import reset_traders
from trading_floor import names, lastnames, model_names
from traders import get_model, get_researcher_tool, MAX_TURNS
from templates import trader_instructions, trade_message, rebalance_message
from mcp_params import trader_mcp_server_params, researcher_mcp_server_params
from accounts_client import read_accounts_resource, read_strategy_resource
from tracers import make_trace_id, LogTracer

## One-time Setup: Pre-cache npm MCP packages

`npx -y` downloads packages on first run and prints npm progress messages to **stdout**.  
The MCP stdio transport reads stdout as JSONRPC — plain text lines cause `Failed to parse JSONRPC message` errors.  
Running the cell below once caches the packages silently, preventing this on all future runs.

In [ ]:
import subprocess

# Pre-warm the npx cache for both MCP packages that use npx.
# capture_output=True silences all npm install chatter — it never reaches the MCP JSONRPC reader.
# These are MCP servers that block on stdin, so TimeoutExpired means 'server started OK'.
_npm_packages = [
    "@modelcontextprotocol/server-brave-search",
    "mcp-memory-libsql",
]
for _pkg in _npm_packages:
    try:
        subprocess.run(["npx", "-y", "--", _pkg],
                       capture_output=True, text=True, timeout=10)
        print(f"  {_pkg}: cached ✓")
    except subprocess.TimeoutExpired:
        # Server started and waited for JSONRPC input — cached and working
        print(f"  {_pkg}: cached ✓ (server started)")

## Addition 1: Direct Web Fetch for Traders

We add `mcp-server-fetch` directly to the Trader's server list so it can independently call `fetch` on any URL. 


In [ ]:
trader_mcp_server_params_extended = trader_mcp_server_params + [
    {"command": "uvx", "args": ["mcp-server-fetch"]},
]

print("Original trader MCP servers:")
for p in trader_mcp_server_params:
    print(f"  {p['command']} {' '.join(p['args'])}")

print("\nExtended trader MCP servers:")
for p in trader_mcp_server_params_extended:
    print(f"  {p['command']} {' '.join(p['args'])}")

---
## Addition 2: Risk Manager Agent

The Risk Manager is a second sub-agent tool available to each Trader.

**What it checks:**
| Rule | Threshold | Rationale |
|---|---|---|
| Concentration limit | ≤ 25% of portfolio per symbol | Prevents catastrophic single-stock loss |
| Cash reserve | ≥ 10% of portfolio always in cash | Ensures liquidity for future opportunities |
| Trade size limit | ≤ 15% of portfolio per single trade | Prevents overcommitting on one decision |

**Returns one of:**
- `VERDICT: APPROVED` trade is within all limits
- `VERDICT: REDUCE` + `SUGGESTED QUANTITY: N` trade is too large, here's a safe quantity
- `VERDICT: REJECTED` trade violates a rule that can't be fixed by resizing


In [ ]:
def risk_manager_instructions() -> str:
    return """
You are a Risk Manager. Your sole job is to review a proposed trade before it is executed.

You will be given:
- The trader's current portfolio: cash balance, current holdings (symbol → quantity), total portfolio value
- The proposed trade: stock symbol, direction (BUY or SELL), and quantity
- The current share price of the stock

Apply these three rules:

1. CONCENTRATION LIMIT: After the trade, no single stock position may exceed 25% of total portfolio value.
   (Total portfolio value = cash + market value of all holdings at current prices.)

2. CASH RESERVE: After a BUY, the remaining cash balance must be at least 10% of total portfolio value.

3. TRADE SIZE LIMIT: The cost of a single BUY trade must not exceed 15% of total portfolio value.

For SELL trades, rules 2 and 3 always pass. Only check rule 1 for sells (you can't be long more than 25% after a sell, but you can't increase a long via a sell anyway, so sells always pass rule 1 as well). Therefore SELLs are always APPROVED.

Respond in EXACTLY this format (no extra text):

VERDICT: APPROVED
REASON: <one sentence>

or

VERDICT: REDUCE
REASON: <one sentence>
SUGGESTED QUANTITY: <integer>

or

VERDICT: REJECTED
REASON: <one sentence>
"""


def risk_manager_tool_description() -> str:
    return (
        "Review a proposed trade against portfolio risk rules before executing it. "
        "Provide: the stock symbol, direction (BUY or SELL), proposed quantity, "
        "current share price, current cash balance, current holdings as a dict of "
        "symbol→quantity, and total portfolio value. "
        "Returns APPROVED, REDUCE (with a suggested safe quantity), or REJECTED with a reason."
    )


async def get_risk_manager(model_name: str) -> Agent:
    """Create the Risk Manager agent (no MCP servers needed — pure reasoning)."""
    return Agent(
        name="RiskManager",
        instructions=risk_manager_instructions(),
        model=get_model(model_name),
    )


async def get_risk_manager_tool(model_name: str) -> Tool:
    """Wrap the Risk Manager as a callable tool for the Trader agent."""
    rm = await get_risk_manager(model_name)
    return rm.as_tool(
        tool_name="RiskManager",
        tool_description=risk_manager_tool_description(),
    )


print("Risk Manager agent factory defined.")

## ExtendedTrader

The `ExtendedTrader` class is a minimal subclass of Ed's `Trader` logic, overriding only two things:

1. **`create_agent`** which adds the Risk Manager tool alongside the Researcher tool
2. **`run_with_mcp_servers`** that uses `trader_mcp_server_params_extended` (which includes `mcp-server-fetch`)

In [ ]:
class ExtendedTrader:
    """
    Extends Ed's Trader with two additions:
      1. Direct web fetch for traders (mcp-server-fetch in trader MCP servers)
      2. Risk Manager sub-agent tool (reviews trades before execution)
    """

    def __init__(self, name: str, lastname: str = "Trader", model_name: str = "gpt-4o-mini"):
        self.name = name
        self.lastname = lastname
        self.model_name = model_name
        self.agent = None
        self.do_trade = True

    async def create_agent(self, trader_mcp_servers, researcher_mcp_servers) -> Agent:
        researcher_tool = await get_researcher_tool(researcher_mcp_servers, self.model_name)
        risk_manager_tool = await get_risk_manager_tool(self.model_name)  # ADDITION 2

        self.agent = Agent(
            name=self.name,
            instructions=trader_instructions(self.name),
            model=get_model(self.model_name),
            tools=[researcher_tool, risk_manager_tool],  # Risk Manager added here
            mcp_servers=trader_mcp_servers,              # Extended list (includes fetch)
        )
        return self.agent

    async def get_account_report(self) -> str:
        account = await read_accounts_resource(self.name)
        account_json = json.loads(account)
        account_json.pop("portfolio_value_time_series", None)
        return json.dumps(account_json)

    async def run_agent(self, trader_mcp_servers, researcher_mcp_servers):
        self.agent = await self.create_agent(trader_mcp_servers, researcher_mcp_servers)
        account = await self.get_account_report()
        strategy = await read_strategy_resource(self.name)
        message = (
            trade_message(self.name, strategy, account)
            if self.do_trade
            else rebalance_message(self.name, strategy, account)
        )
        result = await Runner.run(self.agent, message, max_turns=MAX_TURNS)
        mode = "trading" if self.do_trade else "rebalancing"
        print(f"\n{'='*60}")
        print(f"  {self.name} ({mode})")
        print(f"{'='*60}")
        print(result.final_output)

    async def run_with_mcp_servers(self):
        async with AsyncExitStack() as trader_stack:
            # ADDITION 1: Extended params — trader now also has mcp-server-fetch
            trader_mcp_servers = [
                await trader_stack.enter_async_context(
                    MCPServerStdio(params, client_session_timeout_seconds=120)
                )
                for params in trader_mcp_server_params_extended
            ]
            async with AsyncExitStack() as researcher_stack:
                researcher_mcp_servers = [
                    await researcher_stack.enter_async_context(
                        MCPServerStdio(params, client_session_timeout_seconds=120)
                    )
                    for params in researcher_mcp_server_params(self.name)
                ]
                await self.run_agent(trader_mcp_servers, researcher_mcp_servers)

    async def run_with_trace(self):
        trace_name = f"{self.name}-trading" if self.do_trade else f"{self.name}-rebalancing"
        trace_id = make_trace_id(f"{self.name.lower()}")
        with trace(trace_name, trace_id=trace_id):
            await self.run_with_mcp_servers()

    async def run(self):
        try:
            await self.run_with_trace()
        except Exception as e:
            print(f"Error running trader {self.name}: {e}")
        self.do_trade = not self.do_trade


print("ExtendedTrader defined.")

Quick Check to verify the Risk Manager agent works correctly on its own with a few test cases.

In [ ]:
async def test_risk_manager():
    rm = await get_risk_manager("gpt-4o-mini")

    test_cases = [
        {
            "label": "Safe buy — should be APPROVED",
            "msg": (
                "Proposed trade: BUY 10 shares of AAPL at $200 each = $2,000 total.\n"
                "Current cash: $8,000. Holdings: {}. Total portfolio value: $10,000."
            ),
        },
        {
            "label": "Too large — should be REDUCE",
            "msg": (
                "Proposed trade: BUY 50 shares of NVDA at $100 each = $5,000 total.\n"
                "Current cash: $9,000. Holdings: {'AAPL': 10}. "
                "AAPL price $200 so AAPL value = $2,000. Total portfolio value: $10,000."
            ),
        },
        {
            "label": "Cash reserve breached — should be REJECTED or REDUCE",
            "msg": (
                "Proposed trade: BUY 90 shares of TSLA at $100 each = $9,000 total.\n"
                "Current cash: $9,500. Holdings: {}. Total portfolio value: $10,000."
            ),
        },
        {
            "label": "Sell — should always be APPROVED",
            "msg": (
                "Proposed trade: SELL 5 shares of MSFT at $400 each = $2,000 total.\n"
                "Current cash: $500. Holdings: {'MSFT': 20}. Total portfolio value: $8,500."
            ),
        },
    ]

    for case in test_cases:
        print(f"\n--- {case['label']} ---")
        result = await Runner.run(rm, case["msg"], max_turns=3)
        print(result.final_output)


await test_risk_manager()


Full trading cycle with `ExtendedTrader`. Warren (value investor) is a good choice for the demo because his strategy is conservative and the Risk Manager's constraints align well with value investing principles.

In [ ]:
# Uncomment the line below to reset all four trader accounts to $10,000 and their initial strategies
# reset_traders()

In [ ]:
add_trace_processor(LogTracer())

warren = ExtendedTrader("Warren", lastname="Patience", model_name="gpt-4o-mini")

print("Running Warren (ExtendedTrader) — one trading cycle ...")
print("Tools: Researcher | Risk Manager | direct fetch | accounts | push | market")
print("-" * 60)

await warren.run()

# Show Warren's account after the run
from accounts import Account as _Account
_a = _Account.get("Warren")
_pv = _a.calculate_portfolio_value()
_pnl = _a.calculate_profit_loss(_pv)
print(f"\nWarren's account: cash=${_a.balance:,.0f} | portfolio=${_pv:,.0f} | PnL=${_pnl:+,.0f} | {len(_a.transactions)} trades")

In [ ]:
extended_traders = [
    ExtendedTrader(name, lastname, model_name)
    for name, lastname, model_name in zip(names, lastnames, model_names)
]

print("Starting one round for all four extended traders ...")
print("Each trader has: Researcher | Risk Manager | direct fetch | accounts | push | market")
print("-" * 60)

await asyncio.gather(*[trader.run() for trader in extended_traders])

# Portfolio summary
from accounts import Account
print(f"\n{'='*60}")
print(f"  PORTFOLIO SUMMARY")
print(f"{'='*60}")
print(f"  {'Trader':<10} {'Cash':>10}  {'Port. Value':>12}  {'PnL':>10}  Trades")
print(f"  {'-'*10} {'-'*10}  {'-'*12}  {'-'*10}  ------")
for t in extended_traders:
    a = Account.get(t.name)
    pv = a.calculate_portfolio_value()
    pnl = a.calculate_profit_loss(pv)
    sign = "+" if pnl >= 0 else ""
    print(f"  {t.name:<10} ${a.balance:>9,.0f}  ${pv:>11,.0f}  {sign}${pnl:>9,.0f}  {len(a.transactions)}")

The Risk Manager doesn't force the Trader to follow its advice instead, the Trader's LLM still makes the final call.